In [13]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import KFold
from catboost import CatBoostRanker, Pool
import re
import pymorphy3
import torch
from tqdm import tqdm
import gc
from functools import lru_cache

In [14]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Использую MPS")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Использую CUDA")
else:
    device = torch.device("cpu")
    print("Использую CPU")


Использую MPS


In [15]:
def apk(actual, predicted, k=10):
    if not actual: return 0.0
    if len(predicted) > k: predicted = predicted[:k]
    score, num_hits = 0.0, 0.0
    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    return score / min(len(actual), k)


def mapk(actual, predicted, k=10):
    return np.mean([apk(a, p, k) for a, p in zip(actual, predicted)])

In [16]:
morph = pymorphy3.MorphAnalyzer()


@lru_cache(maxsize=100000)
def lemmatize_word(word):
    return morph.parse(word)[0].normal_form


def clean_html(text):
    if not isinstance(text, str): return ""
    text = BeautifulSoup(text, "html.parser").get_text(separator=" ", strip=True)
    return re.sub(r'\s+', ' ', text).lower()


def clean_query(text):
    text = str(text).lower()
    text = re.sub(r'\b(здравствуйте|добрый|день|подскажите|пожалуйста|привет|вопрос)\b', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def tokenize(text):
    return [lemmatize_word(w) for w in re.findall(r'(?u)\b\w+\b', str(text))]


def get_chunks(title, text, chunk_size=120, overlap=40):
    words = text.split()
    if not words: return [f"passage: {title}"]
    return [f"passage: {title}. {' '.join(words[i:i + chunk_size])}" for i in
            range(0, len(words), chunk_size - overlap)]

In [17]:
print("Загрузка данных...")
articles = pd.read_feather("candidate_public/candidate_data/articles.f")
calibration = pd.read_feather("candidate_public/candidate_data/calibration.f")
test = pd.read_feather("candidate_public/candidate_data/test.f")

articles['clean_body'] = articles['body'].apply(clean_html)
article_ids = articles['article_id'].tolist()
N_DOCS = len(article_ids)

q_calib = calibration['query_text'].apply(clean_query).tolist()
q_test = test['query_text'].apply(clean_query).tolist()
all_queries = q_calib + q_test

Загрузка данных...


In [18]:
print("Обучение BM25 (Заголовки и Текст)...")
tokenized_titles = [tokenize(t) for t in articles['title']]
tokenized_bodies = [tokenize(b) for b in articles['clean_body']]
bm25_t = BM25Okapi(tokenized_titles)
bm25_b = BM25Okapi(tokenized_bodies)

F_bm25_t = np.array([bm25_t.get_scores(tokenize(q)) for q in all_queries])
F_bm25_b = np.array([bm25_b.get_scores(tokenize(q)) for q in all_queries])

Обучение BM25 (Заголовки и Текст)...


In [19]:
print("Обучение TF-IDF (Символы и Леммы)...")
docs_text = (articles['title'] + " " + articles['clean_body']).tolist()
lemmatized_docs = [" ".join(toks) for toks in tokenized_bodies]

# Char TF-IDF
tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2)
char_matrix_docs = tfidf_char.fit_transform(docs_text)
char_matrix_queries = tfidf_char.transform(all_queries)
F_tfidf_c = cosine_similarity(char_matrix_queries, char_matrix_docs)

# Word TF-IDF
tfidf_word = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
word_matrix_docs = tfidf_word.fit_transform(lemmatized_docs)
word_matrix_queries = tfidf_word.transform([" ".join(tokenize(q)) for q in all_queries])
F_tfidf_w = cosine_similarity(word_matrix_queries, word_matrix_docs)

Обучение TF-IDF (Символы и Леммы)...


In [20]:
print("Векторизация BGE-M3...")
bge_model = SentenceTransformer('BAAI/bge-m3', device=device, model_kwargs={"torch_dtype": torch.float16})
bge_model.max_seq_length = 2048

# Body
doc_bge_embs = bge_model.encode(docs_text, batch_size=16, convert_to_tensor=True, show_progress_bar=True, device=device)
q_bge_embs = bge_model.encode(all_queries, batch_size=32, convert_to_tensor=True, show_progress_bar=False,
                              device=device)
F_bge = util.cos_sim(q_bge_embs, doc_bge_embs).cpu().numpy()

# Title
title_bge_embs = bge_model.encode(articles['title'].tolist(), batch_size=32, convert_to_tensor=True, device=device)
F_bge_t = util.cos_sim(q_bge_embs, title_bge_embs).cpu().numpy()

del bge_model, doc_bge_embs, title_bge_embs, q_bge_embs;
gc.collect()
if torch.backends.mps.is_available(): torch.mps.empty_cache()

Векторизация BGE-M3...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

In [21]:
print("Векторизация E5-Large...")
chunk_texts, chunk_doc_idxs = [], []
for idx, row in articles.iterrows():
    for c in get_chunks(row['title'], row['clean_body']):
        chunk_texts.append(c);
        chunk_doc_idxs.append(idx)

e5_model = SentenceTransformer('intfloat/multilingual-e5-large', device=device,
                               model_kwargs={"torch_dtype": torch.float16})
doc_e5_embs = e5_model.encode(chunk_texts, batch_size=64, convert_to_tensor=True, show_progress_bar=True, device=device)
q_e5_texts = [f"query: {q}" for q in all_queries]
q_e5_embs = e5_model.encode(q_e5_texts, batch_size=32, convert_to_tensor=True, show_progress_bar=False, device=device)
sim_e5 = util.cos_sim(q_e5_embs, doc_e5_embs).cpu().numpy()

# Max-Pool и Mean-Pool по чанкам
F_e5_max = np.full((len(all_queries), N_DOCS), -1.0)
F_e5_mean = np.zeros((len(all_queries), N_DOCS))
chunk_counts = np.zeros(N_DOCS)
for c_idx, d_idx in enumerate(chunk_doc_idxs): chunk_counts[d_idx] += 1

for i in range(len(all_queries)):
    for c_idx, score in enumerate(sim_e5[i]):
        d_idx = chunk_doc_idxs[c_idx]
        if score > F_e5_max[i, d_idx]: F_e5_max[i, d_idx] = score
        F_e5_mean[i, d_idx] += score
F_e5_mean = F_e5_mean / np.where(chunk_counts > 0, chunk_counts, 1)

# Title
title_e5_texts = [f"passage: {t}" for t in articles['title']]
title_e5_embs = e5_model.encode(title_e5_texts, batch_size=64, convert_to_tensor=True, device=device)
F_e5_t = util.cos_sim(q_e5_embs, title_e5_embs).cpu().numpy()

del e5_model, doc_e5_embs, title_e5_embs, q_e5_embs;
gc.collect()
if torch.backends.mps.is_available(): torch.mps.empty_cache()

Векторизация E5-Large...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/121 [00:00<?, ?it/s]

In [22]:
print("\nСборка Матрицы Признаков (Hard Negative Mining)...")


# Нормализуем для честной эвристики отбора кандидатов
def norm(x): return (x - np.min(x, axis=1, keepdims=True)) / (np.ptp(x, axis=1, keepdims=True) + 1e-8)


heuristic_scores = norm(F_e5_max) + norm(F_bm25_b) + norm(F_bge)


def build_dataset(query_indices, labels=None, mode='all', top_k=75):
    """
    mode='hnm': Берем только сложные примеры (Top-75) + позитивные (для Обучения)
    mode='all': Берем все 793 статьи (для Валидации/Теста)
    """
    q_ids, d_ids, ys = [], [], []

    for i, q_idx in enumerate(query_indices):
        if mode == 'all':
            docs = np.arange(N_DOCS).tolist()
        else:
            docs = np.argsort(heuristic_scores[q_idx])[::-1][:top_k].tolist()
            if labels is not None:
                for td in labels[i]:
                    if td in article_ids:
                        d_idx = article_ids.index(td)
                        if d_idx not in docs: docs.append(d_idx)

        q_ids.extend([q_idx] * len(docs))
        d_ids.extend(docs)

        if labels is not None:
            truth_idx = [article_ids.index(td) for td in labels[i] if td in article_ids]
            ys.extend([1.0 if d in truth_idx else 0.0 for d in docs])
        else:
            ys.extend([0.0] * len(docs))

    q_ids, d_ids = np.array(q_ids), np.array(d_ids)

    X = pd.DataFrame({
        'bge_score': F_bge[q_ids, d_ids],
        'bge_title': F_bge_t[q_ids, d_ids],
        'e5_max': F_e5_max[q_ids, d_ids],
        'e5_mean': F_e5_mean[q_ids, d_ids],
        'e5_title': F_e5_t[q_ids, d_ids],
        'bm25_title': F_bm25_t[q_ids, d_ids],
        'bm25_body': F_bm25_b[q_ids, d_ids],
        'tfidf_char': F_tfidf_c[q_ids, d_ids],
        'tfidf_word': F_tfidf_w[q_ids, d_ids],
        'q_length': np.array([len(str(all_queries[q]).split()) for q in q_ids])
    })

    return X, np.array(ys), q_ids, d_ids


calib_gt = calibration['ground_truth'].apply(lambda x: [int(i) for i in x.split()]).tolist()
calib_indices = np.arange(len(q_calib))
test_indices = np.arange(len(q_calib), len(all_queries))


Сборка Матрицы Признаков (Hard Negative Mining)...


In [23]:
print("Обучение CatBoostRanker (YetiRank)...")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = []
oof_gt = []

cb_params = {
    'iterations': 1000,
    'depth': 6,
    'learning_rate': 0.05,
    'loss_function': 'YetiRank',
    'eval_metric': 'MAP:top=10',
    'verbose': False,
    'random_seed': 42
}

fold = 1
for train_idx, val_idx in kf.split(calib_indices):
    X_train, y_train, g_train, _ = build_dataset(train_idx, labels=[calib_gt[i] for i in train_idx], mode='hnm')
    train_pool = Pool(X_train, y_train, group_id=g_train)

    X_val, y_val, g_val, d_val = build_dataset(val_idx, labels=[calib_gt[i] for i in val_idx], mode='all')
    val_pool = Pool(X_val, y_val, group_id=g_val)

    model = CatBoostRanker(**cb_params)
    model.fit(train_pool, eval_set=val_pool)

    val_preds = model.predict(val_pool)

    for q_idx in val_idx:
        q_mask = (g_val == q_idx)
        scores = val_preds[q_mask]
        docs = d_val[q_mask]

        top10_docs = docs[np.argsort(scores)[::-1][:10]]
        oof_preds.append([article_ids[d] for d in top10_docs])
        oof_gt.append(calib_gt[q_idx])

    print(f"Fold {fold} (Best MAP@10: {model.get_best_score()['validation']['MAP:top=10']:.4f})")
    fold += 1

print(f"\nMAP@10 НА ВАЛИДАЦИИ: {mapk(oof_gt, oof_preds, k=10):.4f}")

Обучение CatBoostRanker (YetiRank)...
Fold 1 (Best MAP@10: 0.6298)
Fold 2 (Best MAP@10: 0.5623)
Fold 3 (Best MAP@10: 0.5919)
Fold 4 (Best MAP@10: 0.5603)
Fold 5 (Best MAP@10: 0.6328)

MAP@10 НА ВАЛИДАЦИИ: 0.5954


In [24]:
print("\nОбучение финальной модели...")
X_full, y_full, g_full, _ = build_dataset(calib_indices, labels=calib_gt, mode='hnm')
full_train_pool = Pool(X_full, y_full, group_id=g_full)

final_model = CatBoostRanker(**cb_params)
final_model.fit(full_train_pool)


Обучение финальной модели...


CatBoostRanker(depth=6, eval_metric='MAP:top=10', iterations=1000, learning_rate=0.05, loss_function='YetiRank', random_seed=42, verbose=False)

In [25]:
print("\nГенерация ответов...")

X_test, _, g_test, d_test = build_dataset(test_indices, mode='all')
test_preds_raw = final_model.predict(X_test)

test_preds = []
for q_idx in test_indices:
    q_mask = (g_test == q_idx)
    scores = test_preds_raw[q_mask]
    docs = d_test[q_mask]

    top10_docs = docs[np.argsort(scores)[::-1][:10]]
    test_preds.append([article_ids[d] for d in top10_docs])

test_answer = test[['query_id']].copy()
test_answer['answer'] = [" ".join(map(str, preds)) for preds in test_preds]
test_answer.to_csv("answer.csv", index=False)
print("Готово! Файл answer.csv успешно сохранен.")

# Исправленный вывод Feature Importance
importance = final_model.get_feature_importance(data=full_train_pool)
print("\nВажность признаков (Feature Importance):")
for name, val in sorted(zip(X_full.columns, importance), key=lambda x: x[1], reverse=True):
    print(f"{name}: {val:.2f}%")


Генерация ответов...
Готово! Файл answer.csv успешно сохранен.

Важность признаков (Feature Importance):
e5_max: 0.04%
tfidf_word: 0.03%
e5_title: 0.02%
bge_score: 0.02%
tfidf_char: 0.02%
bge_title: 0.02%
bm25_body: 0.01%
e5_mean: 0.01%
bm25_title: 0.01%
q_length: 0.01%
